# Paper Figure 4 — held-out sparse spatial MS-AA decoding (condensed layout)

This revised main-text notebook condenses the held-out sparse-decoding figure.

## New layout
- **Panel A:** all three conditions (**intact, word, rest**) are shown on the **same held-out-decoding plot**.
- **Color** encodes condition:
  - intact = purple
  - word = green
  - rest = black
- **Shade / marker / linestyle** encodes sparse subset size:
  - light = top-3
  - medium = top-5
  - dark = top-10

This replaces the separate top-5 and top-10 panels with a more compact summary while preserving the convergence panel.


### Implementation note

The plotting cell now uses the two tables computed immediately above it:
`summary` for held-out decoding and `joint` for the convergence score. This
avoids relying on guessed variable names and preserves the corrected outer-split
uncertainty calculation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CONDITIONS = ["intact", "word", "rest"]
COLORS = {"intact":"#7B3294", "word":"#2CA25F", "rest":"#222222"}
TOP_M_SHOW = [3, 5, 10]
REPRESENTATIVE_K = 46
T = 300

RUN_CSV = Path(
    "msaa_heldout_topm_outputs_spatial_across/heldout_topm_crossfit_runs.csv"
)

OUTPUT_DIR = Path("outputs/paper_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Figure4_heldout_sparse_MSAA_condensed.png"
FIG_PDF = OUTPUT_DIR / "Figure4_heldout_sparse_MSAA_condensed.pdf"
SOURCE_CSV = OUTPUT_DIR / "Figure4_source_data.csv"

if not RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RUN_CSV}. Run revision_04_heldout_topm_msaa_crossfit first."
    )

run_df = pd.read_csv(RUN_CSV)
print("Cross-fit rows:", len(run_df))

In [ ]:
def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

# A->B and B->A belong to the same outer participant split.
outer_df = (
    run_df
    .groupby(
        ["condition", "K", "top_m", "outer_rep"],
        as_index=False,
    )
    .agg(
        accuracy=("mean", "mean"),
        full_accuracy=("full_mean_same_eval", "mean"),
        retention=("retention_above_chance", "mean"),
    )
)

summary = (
    outer_df
    .groupby(["condition", "K", "top_m"], as_index=False)
    .agg(
        mean=("accuracy", "mean"),
        sem=("accuracy", sem),
        mean_full=("full_accuracy", "mean"),
        mean_retention=("retention", "mean"),
        sem_retention=("retention", sem),
        n_outer=("outer_rep", "size"),
    )
)

summary["chance"] = 1.0 / T

# Joint stimulus score: normalize within condition and m, then take the
# minimum of intact and word. This favors K values that work for both.
stim = summary[summary["condition"].isin(["intact", "word"])].copy()
stim["relative"] = (
    stim["mean"]
    / stim.groupby(["condition", "top_m"])["mean"].transform("max")
)

joint = (
    stim
    .pivot_table(
        index=["K", "top_m"],
        columns="condition",
        values="relative",
        aggfunc="mean",
    )
    .reset_index()
)
joint["joint_score"] = joint[["intact", "word"]].min(axis=1)

summary.to_csv(SOURCE_CSV, index=False)
joint.to_csv(
    OUTPUT_DIR / "Figure4_joint_stimulus_score.csv",
    index=False,
)

display(summary.head())

In [ ]:
# ============================================================
# PAPER FIGURE 4 — CONDENSED LAYOUT
#
# Uses the corrected summaries already computed above:
#   summary = A→B/B→A averaged within outer split, then SEM over outer reps
# ============================================================

from matplotlib.lines import Line2D
from matplotlib.colors import to_rgb


def lighten_color(color, amount=0.5):
    """
    amount=0 -> original color
    amount=1 -> white
    """
    rgb = np.array(to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


# ------------------------------------------------------------
# Visual encoding
# ------------------------------------------------------------
COND_BASE = {
    "intact": "#7B3294",  # purple
    "word":   "#2CA25F",  # green
    "rest":   "#222222",  # black
}

COND_LABEL = {
    "intact": "Intact",
    "word": "Word",
    "rest": "Rest",
}

TOPM_STYLE = {
    3: {
        "lighten": 0.55,
        "linestyle": ":",
        "marker": "o",
        "label": "top-3",
    },
    5: {
        "lighten": 0.28,
        "linestyle": "--",
        "marker": "s",
        "label": "top-5",
    },
    10: {
        "lighten": 0.00,
        "linestyle": "-",
        "marker": "^",
        "label": "top-10",
    },
}

TOP_M_PLOT = [3, 5, 10]


# ------------------------------------------------------------
# Use the corrected summary table created in the previous cell
# ------------------------------------------------------------
sparse_summary = (
    summary[
        summary["top_m"].astype(int).isin(TOP_M_PLOT)
    ]
    .copy()
    .sort_values(
        ["condition", "top_m", "K"]
    )
)

required_sparse = {
    "condition",
    "K",
    "top_m",
    "mean",
    "sem",
}

if not required_sparse.issubset(sparse_summary.columns):
    raise ValueError(
        "summary is missing required columns: "
        f"{required_sparse - set(sparse_summary.columns)}"
    )

print(
    "Sparse top-m values plotted:",
    sorted(sparse_summary["top_m"].astype(int).unique())
)

# Representative K used in the interpretation analysis
anchor_K = REPRESENTATIVE_K
chance_value = 1.0 / T


# ------------------------------------------------------------
# Build intact-only normalized panel
# Normalize within each top-m curve by its own maximum
# ------------------------------------------------------------
intact_norm = (
    sparse_summary[
        sparse_summary["condition"].astype(str) == "intact"
    ]
    .copy()
    .sort_values(["top_m", "K"])
)

if len(intact_norm) == 0:
    raise ValueError("No intact rows found in summary.")

intact_norm["norm_mean"] = np.nan
intact_norm["norm_sem"] = np.nan

for top_m in TOP_M_PLOT:
    mask = intact_norm["top_m"].astype(int) == top_m
    cur = intact_norm.loc[mask].copy()

    if len(cur) == 0:
        continue

    max_mean = cur["mean"].max()

    if max_mean <= 0:
        intact_norm.loc[mask, "norm_mean"] = np.nan
        intact_norm.loc[mask, "norm_sem"] = np.nan
    else:
        intact_norm.loc[mask, "norm_mean"] = cur["mean"] / max_mean
        intact_norm.loc[mask, "norm_sem"] = cur["sem"] / max_mean


# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(13.2, 5.7))

gs = fig.add_gridspec(
    1,
    2,
    width_ratios=[1.72, 1.0],
    wspace=0.30,
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])


# ============================================================
# A — all conditions × top-3/5/10
# ============================================================
for condition in CONDITIONS:

    base_color = COND_BASE[condition]

    for top_m in TOP_M_PLOT:

        cur = (
            sparse_summary[
                (sparse_summary["condition"].astype(str) == condition)
                & (sparse_summary["top_m"].astype(int) == top_m)
            ]
            .sort_values("K")
        )

        if len(cur) == 0:
            continue

        style = TOPM_STYLE[top_m]

        curve_color = lighten_color(
            base_color,
            style["lighten"],
        )

        axA.errorbar(
            cur["K"],
            cur["mean"],
            yerr=cur["sem"],
            color=curve_color,
            linestyle=style["linestyle"],
            marker=style["marker"],
            markersize=4.0,
            linewidth=1.9,
            capsize=2,
            elinewidth=1.0,
            alpha=0.98,
        )

axA.axhline(
    chance_value,
    color="0.60",
    linestyle=":",
    linewidth=1.2,
)

axA.axvline(
    anchor_K,
    color="0.60",
    linestyle="--",
    linewidth=1.2,
)

axA.set_title(
    "Held-out sparse decoding across conditions",
    fontsize=15,
    fontweight="bold",
)

axA.set_xlabel(
    "Model order K",
    fontsize=12,
)

axA.set_ylabel(
    "Timepoint-identification accuracy",
    fontsize=12,
)

axA.tick_params(
    axis="both",
    labelsize=10,
)

axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)

condition_handles = [
    Line2D(
        [0],
        [0],
        color=COND_BASE[c],
        linewidth=2.5,
        label=COND_LABEL[c],
    )
    for c in CONDITIONS
]

subset_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linewidth=1.9,
        linestyle=TOPM_STYLE[m]["linestyle"],
        marker=TOPM_STYLE[m]["marker"],
        markersize=5,
        label=TOPM_STYLE[m]["label"],
    )
    for m in TOP_M_PLOT
]

legend_condition = axA.legend(
    handles=condition_handles,
    title="Condition",
    frameon=False,
    fontsize=9.5,
    title_fontsize=10,
    loc="upper right",
)

axA.add_artist(legend_condition)

axA.legend(
    handles=subset_handles,
    title="Sparse subset",
    frameon=False,
    fontsize=9.5,
    title_fontsize=10,
    loc="center right",
    bbox_to_anchor=(1.0, 0.57),
)


# ============================================================
# B — intact-only normalized panel
# ============================================================
NORM_COLORS = {
    3: "#C2A5CF",  # light purple
    5: "#9970AB",  # medium purple
    10: "#6A3D9A", # darker purple
}

for top_m in TOP_M_PLOT:

    cur = (
        intact_norm[
            intact_norm["top_m"].astype(int) == top_m
        ]
        .sort_values("K")
    )

    if len(cur) == 0:
        continue

    axB.errorbar(
        cur["K"],
        cur["norm_mean"],
        yerr=cur["norm_sem"],
        color=NORM_COLORS[top_m],
        linestyle=TOPM_STYLE[top_m]["linestyle"],
        marker=TOPM_STYLE[top_m]["marker"],
        markersize=5,
        linewidth=2.0,
        capsize=2,
        elinewidth=1.0,
        label=f"top-{top_m}",
    )

axB.axvline(
    anchor_K,
    color="0.60",
    linestyle="--",
    linewidth=1.2,
)

axB.set_ylim(0, 1.05)

axB.set_title(
    "Normalized intact held-out score",
    fontsize=15,
    fontweight="bold",
)

axB.set_xlabel(
    "Model order K",
    fontsize=12,
)

axB.set_ylabel(
    "Normalized intact score",
    fontsize=12,
)

axB.tick_params(
    axis="both",
    labelsize=10,
)

axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

axB.legend(
    frameon=False,
    fontsize=9.5,
    loc="lower right",
)


# ============================================================
# PANEL LABELS
# ============================================================
for ax, letter in zip(
    [axA, axB],
    ["A", "B"],
):
    ax.text(
        -0.09,
        1.04,
        letter,
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        ha="left",
        va="top",
    )

fig.suptitle(
    "Held-out sparse spatial MS-AA decoding",
    fontsize=18,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.09,
    right=0.98,
    bottom=0.14,
    top=0.86,
)

fig.savefig(
    FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

### Figure 4 interpretation

This condensed figure emphasizes the held-out sparse-decoding pattern directly.

Panel A overlays all three conditions on a single axis, using manuscript-consistent
condition colors and shade/marker/linestyle differences to distinguish top-3,
top-5, and top-10 sparse subsets.

Panel B retains the joint intact/word convergence panel to summarize why an
intermediate-scale regime remains a useful focus for downstream interpretation.
